## Cold starts & concurrency

A **cold start** is the extra latency when Lambda has to download your code, start the runtime, and run init before the handler — it happens on the first invocation, after an idle period, or when scaling out to more concurrency. A **warm start** skips all of that and goes straight to the handler. Cold-start cost depends mostly on runtime: **Python/Node ~100–200 ms**, **Java/.NET 500 ms–1 s+** (JVM/CLR startup), container images variable. Reduce the pain by keeping the package small, moving heavy setup into init code (once per environment), choosing a lighter runtime, and — for latency-sensitive surfaces — **Provisioned Concurrency**, which keeps N environments permanently warm (you pay even when idle).

**Concurrency** is the number of execution environments running your function at once. The account has a regional pool (default 1,000) shared by all unreserved functions. **Reserved concurrency** pins a slice to one function — it both **guarantees** that capacity (nothing else can starve it) and **caps** the function there; the classic use is protecting a downstream (cap the Lambda at the database's connection limit). **Provisioned concurrency** is orthogonal — it pre-warms environments to kill cold starts.

When a function exceeds its limit, Lambda **throttles**: synchronous callers get an HTTP **429** immediately, asynchronous events are queued and retried, and event source mappings back off.